# CIPHER — Kaggle Benchmark Notebook
**Calibrated Introspection via Partially Hidden Environment Rules**

Runs the CIPHER metacognition benchmark against frontier LLMs and registers results on Kaggle Benchmarks.

Dataset attached at `/kaggle/input/datasets/wenhaolu49/cipher`.

### Scoring dimensions (all normalized to [0, 1])
| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Objective | 35% | Final plan quality vs oracle beam search |
| Calibration | 25% | Brier score on stated self-knowledge |
| Attention | 20% | Rank correlation on unknown importance |
| Executive | 20% | Plan structure, named risks, alternative plans |

### API keys (Kaggle Secrets)
- `MODEL_PROXY_API_KEY` — auto-injected in Benchmark Task notebooks, covers all Gemini models
- `ANTHROPIC_API_KEY` — optional, for Claude models
- `OPENAI_API_KEY` — optional, for GPT-4o family
- `HF_TOKEN` — optional, for open-source models via HuggingFace

In [ ]:
# 1. Install dependencies
import subprocess, sys

def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

_pip("kaggle-benchmarks")
_pip("google-genai")
_pip("anthropic")
_pip("openai")
_pip("huggingface_hub")
_pip("tqdm")
print("Dependencies installed.")

In [ ]:
# 2. Imports and path setup
import os, sys, json, random, time, datetime
from typing import Any, Dict, List, Optional

_KAGGLE_ROOT = "/kaggle/input/datasets/wenhaolu49/cipher"
_LOCAL_ROOT  = os.path.abspath(os.getcwd())
_LOCAL_PARENT= os.path.abspath(os.path.join(os.getcwd(), ".."))

CIPHER_ROOT = None
for _c in [_KAGGLE_ROOT, _LOCAL_ROOT, _LOCAL_PARENT]:
    if os.path.isdir(os.path.join(_c, "cipher")):
        CIPHER_ROOT = _c
        break
if CIPHER_ROOT is None:
    raise RuntimeError("cipher/ package not found — attach the cipher dataset.")
if CIPHER_ROOT not in sys.path:
    sys.path.insert(0, CIPHER_ROOT)
print(f"cipher root: {CIPHER_ROOT}")

from cipher import build_prompt
from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect, Action
from cipher.schema import validate_response
from cipher.scorer import score_response
from cipher.optimal import oracle_score
print("cipher imports OK.")

In [ ]:
# 3. Configuration
DATA_PATH = next(
    (p for p in [
        os.path.join(_KAGGLE_ROOT, "data", "instances.jsonl"),
        os.path.join(CIPHER_ROOT,  "data", "instances.jsonl"),
    ] if os.path.exists(p)),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError("data/instances.jsonl not found.")

# Set to None for the full 1000-instance submission run
KBENCH_LIMIT = 50

def _secret(name: str) -> str:
    if val := os.environ.get(name, ""):
        return val
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

MODEL_PROXY_API_KEY = _secret("MODEL_PROXY_API_KEY")
ANTHROPIC_API_KEY   = _secret("ANTHROPIC_API_KEY")
OPENAI_API_KEY      = _secret("OPENAI_API_KEY")
HF_TOKEN            = _secret("HF_TOKEN")

# Make key available in env so GoogleGenAI actor can find it
if MODEL_PROXY_API_KEY:
    os.environ["MODEL_PROXY_API_KEY"] = MODEL_PROXY_API_KEY

print(f"DATA_PATH           : {DATA_PATH}")
print(f"KBENCH_LIMIT        : {KBENCH_LIMIT}")
print(f"MODEL_PROXY_API_KEY : {'✓' if MODEL_PROXY_API_KEY else '✗ MISSING'}")
print(f"ANTHROPIC_API_KEY   : {'✓' if ANTHROPIC_API_KEY   else '✗ missing (Claude skipped)'}")
print(f"OPENAI_API_KEY      : {'✓' if OPENAI_API_KEY      else '✗ missing (OpenAI skipped)'}")
print(f"HF_TOKEN            : {'✓' if HF_TOKEN            else '✗ missing (HF models skipped)'}") 

In [ ]:
# 4. Load instances
def _load_jsonl(path: str, limit: Optional[int] = None) -> List[Dict]:
    with open(path) as f:
        recs = [json.loads(line) for line in f if line.strip()]
    return recs[:limit] if limit else recs

def _instance_from_record(rec: Dict[str, Any]) -> Instance:
    h   = rec["hidden"]
    hrl = {e["rule_name"]: set(e["hidden"]) for e in h.get("hidden_fields", [])}
    rules = []
    for r in h["rules"]:
        hides = hrl.get(r["name"], set())
        t, e  = r["trigger"], r["effect"]
        rules.append(Rule(
            name=r["name"],
            trigger=Trigger(kind=t["kind"], i=t["i"], j=t.get("j",-1), k=t.get("k",0),
                            hidden_kind=("trigger_kind" in hides), hidden_k=("trigger_k" in hides)),
            effect=Effect(kind=e["kind"], target=e["target"],
                          delta=e.get("delta",0), source=e.get("source",-1),
                          hidden_kind=("effect_kind" in hides), hidden_delta=("effect_delta" in hides)),
        ))
    initial = State(tuple(EntityState(phase=s["phase"], flux=s["flux"]) for s in h["initial_state"]))
    return Instance(
        id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
        world=World(initial=initial, rules=tuple(rules), horizon=h["horizon"]),
        public_rule_descriptions=[], hidden_fields=h.get("hidden_fields", []),
        metacog_ground_truth=h["metacog_ground_truth"],
        true_unknown_ranking=h["true_unknown_ranking"],
        oracle_objective=h.get("oracle_best"),
    )

KBENCH_RECORDS = _load_jsonl(DATA_PATH, limit=KBENCH_LIMIT)
print(f"Loaded {len(KBENCH_RECORDS)} instances for kbench.")

In [ ]:
# 5. Scoring helper
def _extract_json(text: str) -> Dict[str, Any]:
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return {}
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return {}

def score_text_response(raw_text: str, rec: Dict[str, Any]) -> Dict[str, Any]:
    _zero = dict(composite=0.0, objective=0.0, calibration=0.0, attention=0.0, executive=0.0)
    raw = _extract_json(raw_text)
    if not raw:
        return {**_zero, "error": "no_json"}
    inst = _instance_from_record(rec)
    resp = validate_response(raw)
    bd   = score_response(resp, inst,
                          best_obj=rec["hidden"].get("oracle_best"),
                          worst_obj=rec["hidden"].get("oracle_worst"))
    return bd.to_dict()

In [ ]:
# 6. Stub agents
def stub_noop(inst: Instance) -> Dict[str, Any]:
    return {
        "metacog_assessment": [{"rule_name": g["rule_name"], "component": g["component"],
                                 "known": True, "confidence": 0.5}
                                for g in inst.metacog_ground_truth],
        "critical_unknowns_ranked": [],
        "exploratory_actions": [],
        "final_plan": [{"kind": "wait"}],
        "self_judgment": {"robustness_score": 50, "risks_identified": [],
                          "alternative_if_unknown_X": {}},
    }

def stub_random(inst: Instance) -> Dict[str, Any]:
    rng = random.Random(inst.seed ^ 0xDEADBEEF)
    n   = inst.world.initial.n
    kds = ["pulse", "damp", "shift", "unshift", "observe", "wait"]
    act = lambda: {"kind": rng.choice(kds), "i": rng.randrange(n)}
    return {
        "metacog_assessment": [{"rule_name": g["rule_name"], "component": g["component"],
                                 "known": rng.random() > 0.5, "confidence": rng.random()}
                                for g in inst.metacog_ground_truth],
        "critical_unknowns_ranked": list(inst.true_unknown_ranking[::-1]),
        "exploratory_actions": [act() for _ in range(rng.randint(0, 2))],
        "final_plan":          [act() for _ in range(rng.randint(1, 5))],
        "self_judgment": {"robustness_score": 40, "risks_identified": ["random baseline"],
                          "alternative_if_unknown_X": {"unknown": "", "plan": [act()]}},
    }

def stub_greedy(inst: Instance) -> Dict[str, Any]:
    _, best_plan = oracle_score(inst.world)
    return {
        "metacog_assessment": [{"rule_name": g["rule_name"], "component": g["component"],
                                 "known": True, "confidence": 0.9}
                                for g in inst.metacog_ground_truth],
        "critical_unknowns_ranked": [],
        "exploratory_actions": [],
        "final_plan": [{"kind": a.kind, "i": a.i, "j": a.j} for a in best_plan],
        "self_judgment": {"robustness_score": 70, "risks_identified": [],
                          "alternative_if_unknown_X": {}},
    }

STUB_AGENTS = {"stub-noop": stub_noop, "stub-random": stub_random, "stub-greedy": stub_greedy}
print("Stub agents ready.")

In [ ]:
# 7. Model lists
GEMINI_MODELS = [
    "gemini-2.5-pro",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-1.5-pro",
    "gemini-1.5-flash",
    "gemini-1.5-flash-8b",
]
CLAUDE_MODELS = [
    "claude-opus-4-6",
    "claude-sonnet-4-6",
    "claude-haiku-4-5-20251001",
]
OPENAI_MODELS = ["gpt-4o", "gpt-4o-mini", "o1-mini", "o3-mini"]
HF_MODELS = [
    "meta-llama/Llama-3.3-70B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen2.5-72B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-70B",
]
print("Model lists defined.")

In [ ]:
# 8. kbench task definition
import kaggle_benchmarks as kbench
from kaggle_benchmarks.actors.llms import GoogleGenAI
from tqdm.auto import tqdm

@kbench.task(name="cipher_metacog")
def cipher_task(llm, instance_id: str, prompt: str, record_json: str):
    """
    CIPHER — Calibrated Introspection via Partially Hidden Environment Rules.
    Scored: Objective (35%) · Calibration (25%) · Attention (20%) · Executive (20%).
    Passes when composite >= 0.5.
    """
    reply     = llm.prompt(prompt)
    rec       = json.loads(record_json)
    breakdown = score_text_response(str(reply), rec)
    composite = breakdown.get("composite", 0.0)
    kbench.assertions.assert_true(
        composite >= 0.5,
        f"[{instance_id}] composite {composite:.3f} < 0.5 "
        f"obj={breakdown.get('objective',0):.3f} "
        f"cal={breakdown.get('calibration',0):.3f} "
        f"att={breakdown.get('attention',0):.3f} "
        f"exe={breakdown.get('executive',0):.3f}",
    )

print(f"cipher_task defined — {len(KBENCH_RECORDS)} instances per model.")

In [ ]:
# 9. Actor wrappers for non-Gemini models

class _ClaudeActor:
    def __init__(self, model_id: str):
        import anthropic
        self._client   = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        self._model_id = model_id
    def prompt(self, text: str) -> str:
        msg = self._client.messages.create(
            model=self._model_id, max_tokens=2048,
            messages=[{"role": "user", "content": text}])
        return "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")

class _OpenAIActor:
    def __init__(self, model_id: str):
        from openai import OpenAI
        self._client    = OpenAI(api_key=OPENAI_API_KEY)
        self._model_id  = model_id
        self._reasoning = model_id.startswith(("o1", "o3"))
    def prompt(self, text: str) -> str:
        kwargs: Dict[str, Any] = {"model": self._model_id, "max_completion_tokens": 2048,
                                   "messages": [{"role": "user", "content": text}]}
        if not self._reasoning:
            kwargs["temperature"] = 0.0
        return self._client.chat.completions.create(**kwargs).choices[0].message.content or ""

class _HFActor:
    def __init__(self, model_id: str):
        from huggingface_hub import InferenceClient
        self._client = InferenceClient(model=model_id, token=HF_TOKEN)
    def prompt(self, text: str) -> str:
        resp = self._client.chat_completion(
            messages=[{"role": "user", "content": text}], max_tokens=2048, temperature=0.0)
        return resp.choices[0].message.content or ""

class _StubActor:
    def __init__(self, stub_fn):
        self._fn  = stub_fn
        self._rec: Optional[Dict] = None
    def set_rec(self, rec: Dict) -> None:
        self._rec = rec
    def prompt(self, _text: str) -> str:
        if self._rec is None:
            return "{}"
        return json.dumps(self._fn(_instance_from_record(self._rec)))

print("Actor wrappers defined.")

In [ ]:
# 10. Run kbench: Gemini
# GoogleGenAI reads MODEL_PROXY_API_KEY from the environment automatically.
# Do NOT pass a genai.Client manually — the proxy only works via the actor.

if not MODEL_PROXY_API_KEY:
    print("Skipping — MODEL_PROXY_API_KEY not set.")
else:
    for _mid in GEMINI_MODELS:
        print(f"\nkbench: {_mid} ({len(KBENCH_RECORDS)} instances)")
        _llm = GoogleGenAI(model=_mid)
        for _rec in tqdm(KBENCH_RECORDS, desc=_mid, leave=False):
            try:
                cipher_task.run(llm=_llm, instance_id=_rec["id"],
                                prompt=_rec["prompt"], record_json=json.dumps(_rec))
            except Exception as _e:
                print(f"  warning {_rec['id']}: {_e}")
        print("  done")
    print("\nGemini kbench complete.")

In [ ]:
# 11. Run kbench: Claude
if not ANTHROPIC_API_KEY:
    print("Skipping — ANTHROPIC_API_KEY not set.")
else:
    for _mid in CLAUDE_MODELS:
        print(f"\nkbench: {_mid} ({len(KBENCH_RECORDS)} instances)")
        _llm = _ClaudeActor(_mid)
        for _rec in tqdm(KBENCH_RECORDS, desc=_mid, leave=False):
            try:
                cipher_task.run(llm=_llm, instance_id=_rec["id"],
                                prompt=_rec["prompt"], record_json=json.dumps(_rec))
            except Exception as _e:
                print(f"  warning {_rec['id']}: {_e}")
        print("  done")
    print("\nClaude kbench complete.")

In [ ]:
# 12. Run kbench: OpenAI
if not OPENAI_API_KEY:
    print("Skipping — OPENAI_API_KEY not set.")
else:
    for _mid in OPENAI_MODELS:
        print(f"\nkbench: {_mid} ({len(KBENCH_RECORDS)} instances)")
        _llm = _OpenAIActor(_mid)
        for _rec in tqdm(KBENCH_RECORDS, desc=_mid, leave=False):
            try:
                cipher_task.run(llm=_llm, instance_id=_rec["id"],
                                prompt=_rec["prompt"], record_json=json.dumps(_rec))
            except Exception as _e:
                print(f"  warning {_rec['id']}: {_e}")
        print("  done")
    print("\nOpenAI kbench complete.")

In [ ]:
# 13. Run kbench: HuggingFace
if not HF_TOKEN:
    print("Skipping — HF_TOKEN not set.")
else:
    for _mid in HF_MODELS:
        _short = _mid.split("/")[-1]
        print(f"\nkbench: {_short} ({len(KBENCH_RECORDS)} instances)")
        _llm = _HFActor(_mid)
        for _rec in tqdm(KBENCH_RECORDS, desc=_short, leave=False):
            try:
                cipher_task.run(llm=_llm, instance_id=_rec["id"],
                                prompt=_rec["prompt"], record_json=json.dumps(_rec))
            except Exception as _e:
                print(f"  warning {_rec['id']}: {_e}")
        print("  done")
    print("\nHF kbench complete.")

In [ ]:
# 14. Run kbench: stubs
for _sname, _sfn in STUB_AGENTS.items():
    print(f"\nkbench: {_sname} ({len(KBENCH_RECORDS)} instances)")
    _llm = _StubActor(_sfn)
    for _rec in tqdm(KBENCH_RECORDS, desc=_sname, leave=False):
        _llm.set_rec(_rec)
        try:
            cipher_task.run(llm=_llm, instance_id=_rec["id"],
                            prompt=_rec["prompt"], record_json=json.dumps(_rec))
        except Exception:
            pass
    print("  done")
print("\nAll kbench runs complete.")